# Session 15: Conditional probability and Bayes (Python lab)

You will read conditional probabilities from a table of counts, test independence, update a belief with Bayes theorem, and build a small Naive Bayes spam filter from word counts.

**How to work.** In each step: read the idea, write a PREDICT guess, run the cell, READ the output, then CHANGE one thing in the experiment cell. Cells with `None  # your code here` are yours to fill. The automatic checks print OK or NOT YET. Run the cells from the top, in order. All numbers are illustrative (made up to show the arithmetic).

---
## Step 0: Set up

Only numpy is needed.

In [ ]:
import numpy as np
print("ready")

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks(total=None):
    # total = how many checks this notebook has; a skipped or crashed cell still counts as NOT YET
    k = sum(_results.values()); n = max(total or 0, len(_results))
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: A table of counts: joint, marginal and conditional

100 emails. Rows: spam, ham. Columns: contains "free", does not contain "free".

**PREDICT first.** Is P(spam | free) bigger or smaller than P(spam) = 0.4? Is P(free | spam) the same number as P(spam | free)?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
table = np.array([[28, 12], [6, 54]])        # rows spam, ham; columns free, no free
total = table.sum()
# Hint: the conditional probability is a joint probability divided by a marginal one. Column 0 of the table is the emails with "free".
p_joint = None  # your code here
p_free = None  # your code here
p_spam_given_free = None  # your code here
p_free_given_spam = None  # your code here
check("P(spam and free)", p_joint, 0.28, 0.0001)
check("P(free)", p_free, 0.34, 0.0001)
check("P(spam | free)", p_spam_given_free, 0.8235, 0.0001)
check("P(free | spam)", p_free_given_spam, 0.70, 0.0001)

**READ the output.** Why are P(spam | free) and P(free | spam) different numbers? Which groups of emails does each one divide by?

> My answer: 

**CHANGE one thing.** Run the experiment cell: P(ham | free). Predict it first. It must add up with P(spam | free) to 1.

> What I saw: 

In [ ]:
print("P(ham | free) =", round(6 / 34, 4))

---
## Step 2: Independence (recall Session 13)

Independent events satisfy P(A and B) = P(A) × P(B). If "free" and spam were independent, the count of spam emails with "free" would be total × P(spam) × P(free).

**PREDICT first.** Will the expected count be bigger or smaller than the observed 28?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: the independence rule from the slides, scaled up to all the emails. P(spam) = 0.4.
expected = None  # your code here
check("expected count if independent", expected, 13.6, 0.0001)

**READ the output.** Observed is much larger than expected. What does that say about the word "free" and spam?

> My answer: 

---
## Step 3: Bayes theorem with a medical test

Prior P(H) = 0.01. Likelihood (sensitivity) P(positive | sick) = 0.90. False positive rate P(positive | healthy) = 0.10.

**PREDICT first.** After one positive test, will the chance of being sick be about 90%, 50% or under 10%?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
def bayes(prior, like, fpr):
    # Hint: the evidence adds up both routes to a positive test (sick and positive, healthy and positive). The posterior is the sick-and-positive route divided by the evidence.
    return None, None  # your code here
post1, p_e = bayes(0.01, 0.90, 0.10)
post2 = None  # your code here: use post1 as the new prior
check("P(positive), the evidence", p_e, 0.108, 0.0001)
check("posterior after one positive test", post1, 0.0833, 0.0001)
check("posterior after two positive tests", post2, 0.45, 0.0001)

**READ the output.** Most positive results come from healthy people. How many of 1,000 people are in each group (the tree on the slides)?

> My answer: 

**CHANGE one thing.** Run the experiment cell: a prevalence of 10% instead of 1%. Predict the posterior first.

> What I saw: 

In [ ]:
print(round(0.9 * 0.1 / (0.9 * 0.1 + 0.1 * 0.9), 4))

---
## Step 4: Naive Bayes spam filter from counts

Counts: how many of the 40 spam and 60 ham emails contain the words free, prize and meeting. Smoothed likelihood = (count + 1) / (class size + 2).

**PREDICT first.** Email 1 has "free" and "prize" but not "meeting". Email 2 has "free" and "meeting" but not "prize". Which is more likely spam?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
spam_counts = np.array([28, 20, 4])      # free, prize, meeting
ham_counts = np.array([6, 0, 30])
n_spam, n_ham = 40, 60
email1 = np.array([1, 1, 0])           # 1 = the word is in the email
email2 = np.array([1, 0, 1])

In [ ]:
def posterior(has, smooth=True):
    add = 1 if smooth else 0                                   # add = 1 smooths; add = 0 uses plain counts
    like_s = (spam_counts + add) / (n_spam + 2 * add)          # likelihood of each word in spam
    like_h = None  # your code here: the same line for ham
    score_s = (n_spam / 100) * np.prod(np.where(has == 1, like_s, 1 - like_s))   # prior times one factor per word
    score_h = None  # your code here: the same line for ham
    if like_h is None or score_h is None:
        return None
    return score_s / (score_s + score_h)
# Hint: copy the spam lines and change spam to ham (60 ham emails). Then fill the two single likelihoods below by hand: (count + 1) / (class size + 2).
like_free_spam = None  # your code here
like_prize_ham = None  # your code here
p1, p2 = posterior(email1), posterior(email2)
check("smoothed likelihood of free in spam", like_free_spam, 0.6905, 0.0001)
check("smoothed likelihood of prize in ham", like_prize_ham, 0.0161, 0.0001)
check("posterior spam probability of email 1", p1, 0.9955, 0.0001)
check("posterior spam probability of email 2", p2, 0.3304, 0.0001)

In [ ]:
# Long emails: add logarithms (no underflow), then the two-class softmax, which is the sigmoid of the gap (Session 11).
log_s = np.log(0.4) + np.log(np.where(email1 == 1, (spam_counts + 1) / 42, 1 - (spam_counts + 1) / 42)).sum()   # given: the spam log score
# Hint: copy the line above for ham (prior 0.6, 60 ham emails, so divide by 62). Then p_log = 1 / (1 + np.exp(log_h - log_s)).
log_h = None  # your code here
p_log = None  # your code here
check("log score of ham for email 1", log_h, -7.5123, 0.0005)
check("posterior of email 1 computed with logs", p_log, 0.9955, 0.0001)

**READ the output.** Which words pushed email 2 towards ham? Is the filter sure about it?

> My answer: 

---
## Step 5: Break it: remove the smoothing

Set smooth=False so the zero count of "prize" in ham stays zero.

**PREDICT first.** What will the posterior of email 1 become if the ham likelihood of "prize" is exactly 0?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: use your posterior function on email 1 with the smoothing switched off.
p1_raw = None  # your code here
check("posterior of email 1 without smoothing", p1_raw, 1.0, 0.0001)

**READ the output.** The filter now says 100% spam from three words. Is that sensible? What does it say about reading a stated probability from any model?

> My answer: 

**CHANGE one thing.** Run the experiment cell: the effect of the prior. Use a prior of 0.01 for spam (a rare-spam mailbox) with the smoothed likelihoods of email 1 and see how far the posterior moves.

> What I saw: 

In [ ]:
ls = (spam_counts + 1) / 42
lh = (ham_counts + 1) / 62
ss = 0.01 * np.prod(np.where(email1 == 1, ls, 1 - ls))
sh = 0.99 * np.prod(np.where(email1 == 1, lh, 1 - lh))
print(round(ss / (ss + sh), 4))

---
## Step 6: Show the checks

Run this last cell. A learner who has filled every answer sees "All 15 checks pass".

In [ ]:
show_checks(15)